## Assignment 3 

### Task 1 - Data Ingestion

In [1]:
from pathlib import Path
import pandas as pd
import dask.dataframe as dd
import geopandas as gpd

In [ ]:

DATA_DIR = Path("/d/hpc/projects/FRI/bigdata/data/Taxi")

files = sorted(DATA_DIR.rglob("yellow_tripdata_*.parquet"))
print("Total yellow taxi parquet files:", len(files))
print("First few files:")
for f in files[:10]:
    print(f.name)


Total yellow taxi parquet files: 193
First few files:
yellow_tripdata_2009-01.parquet
yellow_tripdata_2009-02.parquet
yellow_tripdata_2009-03.parquet
yellow_tripdata_2009-04.parquet
yellow_tripdata_2009-05.parquet
yellow_tripdata_2009-06.parquet
yellow_tripdata_2009-07.parquet
yellow_tripdata_2009-08.parquet
yellow_tripdata_2009-09.parquet
yellow_tripdata_2009-10.parquet


In [6]:
files_by_year = {}
for f in files:
    year = f.stem.split("_")[-1][:4]
    files_by_year.setdefault(year, []).append(str(f))

print("Available years:", sorted(files_by_year.keys()))


Available years: ['2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [7]:
selected_years = ["2019", "2020", "2021", "2022", "2023"]
for year in selected_years:
    df = dd.read_parquet(files_by_year[year])
    print(f"\n===== YEAR {year} =====")
    print("Number of columns:", len(df.columns))
    print(df.dtypes.astype(str))


===== YEAR 2019 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                      object
dtype: str

===== YEAR 2020 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count    

In [19]:
target_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]


In [20]:
def normalize_partition(pdf):
    pdf = pdf.copy()

    for col in target_columns:
        if col not in pdf.columns:
            pdf[col] = pd.NA

    pdf = pdf[target_columns]

    numeric_cols = [
        "VendorID", "passenger_count", "trip_distance", "RatecodeID",
        "PULocationID", "DOLocationID", "payment_type",
        "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "total_amount", "congestion_surcharge",
        "airport_fee"
    ]

    for col in numeric_cols:
        pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

    pdf["store_and_fwd_flag"] = pdf["store_and_fwd_flag"].astype("string")
    pdf["tpep_pickup_datetime"] = pd.to_datetime(pdf["tpep_pickup_datetime"], errors="coerce")
    pdf["tpep_dropoff_datetime"] = pd.to_datetime(pdf["tpep_dropoff_datetime"], errors="coerce")

    return pdf


In [22]:
meta = pd.DataFrame({
    "VendorID": pd.Series(dtype="float64"),
    "tpep_pickup_datetime": pd.Series(dtype="datetime64[ns]"),
    "tpep_dropoff_datetime": pd.Series(dtype="datetime64[ns]"),
    "passenger_count": pd.Series(dtype="float64"),
    "trip_distance": pd.Series(dtype="float64"),
    "RatecodeID": pd.Series(dtype="float64"),
    "store_and_fwd_flag": pd.Series(dtype="string"),
    "PULocationID": pd.Series(dtype="float64"),
    "DOLocationID": pd.Series(dtype="float64"),
    "payment_type": pd.Series(dtype="float64"),
    "fare_amount": pd.Series(dtype="float64"),
    "extra": pd.Series(dtype="float64"),
    "mta_tax": pd.Series(dtype="float64"),
    "tip_amount": pd.Series(dtype="float64"),
    "tolls_amount": pd.Series(dtype="float64"),
    "improvement_surcharge": pd.Series(dtype="float64"),
    "total_amount": pd.Series(dtype="float64"),
    "congestion_surcharge": pd.Series(dtype="float64"),
    "airport_fee": pd.Series(dtype="float64"),
    "year": pd.Series(dtype="int64"),
})


In [23]:
normalized_dfs = {}

for year in selected_years:
    monthly_dfs = []

    for file in files_by_year[year]:
        df_month = dd.read_parquet(file)
        df_month = df_month.map_partitions(normalize_partition, meta=meta.drop(columns=["year"]))
        monthly_dfs.append(df_month)

    df_year = dd.concat(monthly_dfs)
    df_year = df_year.assign(year=int(year))
    normalized_dfs[year] = df_year

    print(year, "normalized")



2019 normalized
2020 normalized
2021 normalized
2022 normalized
2023 normalized


In [26]:
OUTPUT_DIR = Path("/d/hpc/home/bv7063/bigdata/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for year in selected_years:
    out_dir = OUTPUT_DIR / f"csv_{year}"
    out_dir.mkdir(parents=True, exist_ok=True)

    normalized_dfs[year].to_csv(
        str(out_dir / "part-*.csv"),
        index=False
    )

    print(f"{year} CSV written to {out_dir}")


2019 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2019
2020 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2020
2021 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2021
2022 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2022
2023 CSV written to /d/hpc/home/bv7063/bigdata/output/csv_2023


In [ ]:
# number of partitions per year
for year in selected_years:
    print(year, normalized_dfs[year].npartitions)


2019 12
2020 12
2021 12
2022 12
2023 12


In [30]:
## HDF5 VS CSV
one_year = "2023"
one_year_df = normalized_dfs[one_year]

one_year_csv_dir = OUTPUT_DIR / f"one_year_csv_{one_year}"
one_year_csv_dir.mkdir(parents=True, exist_ok=True)

one_year_pd = one_year_df.compute()
one_year_pd.to_hdf(
    OUTPUT_DIR / f"{one_year}.h5",
    key="taxi",
    mode="w",
    format="table"
)

one_year_df.to_csv(
    str(one_year_csv_dir / "part-*.csv"),
    index=False
)


['/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-00.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-01.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-02.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-03.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-04.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-05.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-06.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-07.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-08.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-09.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-10.csv',
 '/d/hpc/home/bv7063/bigdata/output/one_year_csv_2023/part-11.csv']

In [ ]:
# combined parquet partion by year
combined_df = dd.concat([normalized_dfs[year] for year in selected_years])
combined_df = combined_df.repartition(partition_size="256MB")
optimized_parquet_dir = OUTPUT_DIR / "optimized_parquet"

combined_df.to_parquet(
    optimized_parquet_dir,
    engine="pyarrow",
    write_index=False,
    partition_on=["year"],
    row_group_size=100_000
)

# partion size means approximate size of each dask partion 
# row group size means within one dask partion / parquet file how are the rows goruped internally  
# dask partion and parquet file is not the same and can differ but durign writing one dask partion often becomes one parquest file

### Task 2 - Comparing Storage Formats


In [34]:
def get_size_bytes(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

def bytes_to_gb(x):
    return x / (1024**3)


In [44]:
# 5-year original parquet size
original_parquet_size = sum(
    Path(f).stat().st_size
    for year in selected_years
    for f in files_by_year[year]
)

# 5-year total CSV size
total_csv_5y = sum(
    get_size_bytes(OUTPUT_DIR / f"csv_{year}")
    for year in selected_years
)

# 5-year optimized parquet size
optimized_parquet_size = get_size_bytes(OUTPUT_DIR / "optimized_parquet")

comparison_five_years = pd.DataFrame([
    {"format": "Original parquet (5 years)", "size_bytes": original_parquet_size},
    {"format": "CSV total (5 years)", "size_bytes": total_csv_5y},
    {"format": "Optimized parquet (5 years)", "size_bytes": optimized_parquet_size},
])

comparison_five_years["size_gb"] = comparison_five_years["size_bytes"].apply(bytes_to_gb)
comparison_five_years = comparison_five_years.sort_values("size_gb")

print("Five-year comparison")
display(comparison_five_years)


Five-year comparison


,format,size_bytes,size_gb
0,Original parquet (5 years),3348556571,3.118586
2,Optimized parquet (5 years),4246213938,3.954595
1,CSV total (5 years),23447058209,21.836775


In [45]:
# 1-year comparison
one_year_original_parquet_size = sum(
    Path(f).stat().st_size for f in files_by_year[one_year]
)

one_year_csv_size = get_size_bytes(OUTPUT_DIR / f"one_year_csv_{one_year}")
one_year_hdf5_size = get_size_bytes(OUTPUT_DIR / f"{one_year}.h5")

comparison_one_year = pd.DataFrame([
    {"format": f"Original parquet ({one_year})", "size_bytes": one_year_original_parquet_size},
    {"format": f"CSV ({one_year})", "size_bytes": one_year_csv_size},
    {"format": f"HDF5 ({one_year})", "size_bytes": one_year_hdf5_size},
])

comparison_one_year["size_gb"] = comparison_one_year["size_bytes"].apply(bytes_to_gb)
comparison_one_year = comparison_one_year.sort_values("size_gb")

print(f"One-year comparison ({one_year})")
display(comparison_one_year)

One-year comparison (2023)


,format,size_bytes,size_gb
0,Original parquet (2023),635743618,0.592082
1,CSV (2023),4106528575,3.824503
2,HDF5 (2023),6296110531,5.863710


### Task 3 - Data Augmentation

#### Weather Data 

In [4]:
BASE_DIR = Path("/d/hpc/home/bv7063/bigdata/output")
taxi_df = dd.read_parquet(BASE_DIR / "optimized_parquet")
taxi_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,year
0,1.0,2019-01-01 00:46:40,2019-01-01 00:53:20,1.0,1.5,1.0,N,151.0,239.0,1.0,7.0,0.5,0.5,1.65,0.0,0.3,9.95,NaN,NaN,2019
1,1.0,2019-01-01 00:59:47,2019-01-01 01:18:59,1.0,2.6,1.0,N,239.0,246.0,1.0,14.0,0.5,0.5,1.00,0.0,0.3,16.30,NaN,NaN,2019
2,2.0,2018-12-21 13:48:30,2018-12-21 13:52:40,3.0,0.0,1.0,N,236.0,236.0,1.0,4.5,0.5,0.5,0.00,0.0,0.3,5.80,NaN,NaN,2019
3,2.0,2018-11-28 15:52:25,2018-11-28 15:55:45,5.0,0.0,1.0,N,193.0,193.0,2.0,3.5,0.5,0.5,0.00,0.0,0.3,7.55,NaN,NaN,2019
4,2.0,2018-11-28 15:56:57,2018-11-28 15:58:33,5.0,0.0,2.0,N,193.0,193.0,2.0,52.0,0.0,0.5,0.00,0.0,0.3,55.55,NaN,NaN,2019


In [5]:
### analyze wether data overlap is between year 2019 and 2022
weather = pd.read_csv("/d/hpc/home/bv7063/bigdata/NYC_Weather_2016_2022.csv")

weather["time"] = pd.to_datetime(weather["time"], errors="coerce")
weather = weather.dropna(subset=["time", "temperature_2m (°C)", "precipitation (mm)"])

weather["weather_date"] = weather["time"].dt.date
weather["weather_hour"] = weather["time"].dt.hour

weather = weather.rename(columns={
    "temperature_2m (°C)": "temperature_c",
    "precipitation (mm)": "precipitation_mm",
    "rain (mm)": "rain_mm",
    "cloudcover (%)": "cloudcover_pct",
    "windspeed_10m (km/h)": "windspeed_kmh",
})

weather[["time", "weather_date", "weather_hour", "temperature_c", "precipitation_mm"]].head()


,time,weather_date,weather_hour,temperature_c,precipitation_mm
0,2016-01-01 00:00:00,2016-01-01,0,7.6,0.0
1,2016-01-01 01:00:00,2016-01-01,1,7.5,0.0
2,2016-01-01 02:00:00,2016-01-01,2,7.1,0.0
3,2016-01-01 03:00:00,2016-01-01,3,6.6,0.0
4,2016-01-01 04:00:00,2016-01-01,4,6.3,0.0


In [6]:
weather_small = weather[
    ["weather_date", "weather_hour", "temperature_c", "precipitation_mm", "rain_mm", "cloudcover_pct", "windspeed_kmh"]
].copy()


In [7]:
# join based on pick up time
BASE_DIR = Path("/d/hpc/home/bv7063/bigdata/output")

taxi_df = dd.read_parquet(BASE_DIR / "optimized_parquet")
taxi_df = taxi_df[taxi_df["year"].isin([2019, 2020, 2021, 2022])]

taxi_df = taxi_df.assign(
    pickup_date=taxi_df["tpep_pickup_datetime"].dt.date,
    pickup_hour=taxi_df["tpep_pickup_datetime"].dt.hour,
)

weather = pd.read_csv("/d/hpc/home/bv7063/bigdata/NYC_Weather_2016_2022.csv")

weather["time"] = pd.to_datetime(weather["time"], errors="coerce")
weather = weather.dropna(subset=["time", "temperature_2m (°C)", "precipitation (mm)"])

weather["pickup_date"] = weather["time"].dt.date
weather["pickup_hour"] = weather["time"].dt.hour

weather = weather.rename(columns={
    "temperature_2m (°C)": "temperature_c",
    "precipitation (mm)": "precipitation_mm",
    "rain (mm)": "rain_mm",
    "cloudcover (%)": "cloudcover_pct",
    "windspeed_10m (km/h)": "windspeed_kmh",
})

In [15]:
taxi_df = dd.read_parquet(BASE_DIR / "optimized_parquet")

taxi_df = taxi_df[
    (taxi_df["tpep_pickup_datetime"] >= "2019-01-01") &
    (taxi_df["tpep_pickup_datetime"] < "2022-10-26")
]

taxi_df = taxi_df.assign(
    pickup_time_hour=taxi_df["tpep_pickup_datetime"].dt.floor("h")
)


In [16]:
weather = pd.read_csv("/d/hpc/home/bv7063/bigdata/NYC_Weather_2016_2022.csv")

weather["pickup_time_hour"] = pd.to_datetime(weather["time"], errors="coerce")
weather = weather.dropna(subset=["pickup_time_hour", "temperature_2m (°C)", "precipitation (mm)"])

weather = weather.rename(columns={
    "temperature_2m (°C)": "temperature_c",
    "precipitation (mm)": "precipitation_mm",
    "rain (mm)": "rain_mm",
    "cloudcover (%)": "cloudcover_pct",
    "windspeed_10m (km/h)": "windspeed_kmh",
})

weather_small = weather[
    [
        "pickup_time_hour",
        "temperature_c",
        "precipitation_mm",
        "rain_mm",
        "cloudcover_pct",
        "windspeed_kmh",
    ]
].drop_duplicates()


In [17]:
weather_dd = dd.from_pandas(weather_small, npartitions=1)


In [18]:
taxi_weather_df = taxi_df.merge(
    weather_dd,
    on="pickup_time_hour",
    how="left"
)


/d/hpc/home/bv7063/bigdata/.conda-env/lib/python3.11/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+------------------------------------------+----------------+----------------+
| Merge columns                            | left dtype     | right dtype    |
+------------------------------------------+----------------+----------------+
| ('pickup_time_hour', 'pickup_time_hour') | datetime64[ns] | datetime64[us] |
+------------------------------------------+----------------+----------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(


In [23]:
"""
After joining hourly weather data by pickup timestamp rounded to the hour, 
approximately 99.5% of taxi trips were successfully matched. 
A small fraction remained unmatched due to timestamp anomalies 
or missing weather observations.
"""

taxi_weather_df[[
    "temperature_c",
    "precipitation_mm",
    "rain_mm",
    "cloudcover_pct",
    "windspeed_kmh"
]].isna().mean().compute()



# Number of missing rows
#missing_weather_rows = taxi_weather_df["temperature_c"].isna().sum().compute()
#total_rows = len(taxi_weather_df)
#missing_weather_rows, total_rows



temperature_c       0.004866
precipitation_mm    0.004866
rain_mm             0.004866
cloudcover_pct      0.004866
windspeed_kmh       0.004866
dtype: float64

In [20]:
taxi_weather_df.to_parquet(
    BASE_DIR / "augmented_weather_parquet",
    engine="pyarrow",
    write_index=False
)


In [29]:
taxi_weather_df[[
    "tpep_pickup_datetime",
    "pickup_time_hour",
    "PULocationID",
    "DOLocationID",
    "temperature_c",
    "precipitation_mm",
    "rain_mm",
    "cloudcover_pct",
    "windspeed_kmh",
]].head(10)


,tpep_pickup_datetime,pickup_time_hour,PULocationID,DOLocationID,temperature_c,precipitation_mm,rain_mm,cloudcover_pct,windspeed_kmh
0,2019-01-01 00:46:40,2019-01-01,151.0,239.0,6.4,1.8,1.8,100.0,14.4
1,2019-01-01 00:59:47,2019-01-01,239.0,246.0,6.4,1.8,1.8,100.0,14.4
2,2019-01-01 00:21:28,2019-01-01,163.0,229.0,6.4,1.8,1.8,100.0,14.4
3,2019-01-01 00:32:01,2019-01-01,229.0,7.0,6.4,1.8,1.8,100.0,14.4
4,2019-01-01 00:57:32,2019-01-01,141.0,234.0,6.4,1.8,1.8,100.0,14.4
5,2019-01-01 00:24:04,2019-01-01,246.0,162.0,6.4,1.8,1.8,100.0,14.4
6,2019-01-01 00:21:59,2019-01-01,238.0,151.0,6.4,1.8,1.8,100.0,14.4
7,2019-01-01 00:45:21,2019-01-01,163.0,25.0,6.4,1.8,1.8,100.0,14.4
8,2019-01-01 00:43:19,2019-01-01,224.0,25.0,6.4,1.8,1.8,100.0,14.4
9,2019-01-01 00:58:24,2019-01-01,141.0,234.0,6.4,1.8,1.8,100.0,14.4


In [30]:
taxi_weather_df[[
    "pickup_time_hour",
    "temperature_c",
    "precipitation_mm",
    "rain_mm",
    "cloudcover_pct",
    "windspeed_kmh",
]].drop_duplicates().compute().sort_values("pickup_time_hour").head(30)


,pickup_time_hour,temperature_c,precipitation_mm,rain_mm,cloudcover_pct,windspeed_kmh
0,2019-01-01 00:00:00,6.4,1.8,1.8,100.0,14.4
756,2019-01-01 01:00:00,6.6,1.7,1.7,100.0,16.2
13267,2019-01-01 02:00:00,6.8,2.4,2.4,100.0,17.3
29137,2019-01-01 03:00:00,7.6,4.1,4.1,100.0,18.7
30004,2019-01-01 04:00:00,7.9,5.6,5.6,100.0,14.0
52398,2019-01-01 05:00:00,8.1,5.1,5.1,100.0,8.5
59383,2019-01-01 06:00:00,8.5,3.3,3.3,100.0,11.3
63111,2019-01-01 07:00:00,9.1,0.4,0.4,100.0,20.5
65501,2019-01-01 08:00:00,9.8,0.0,0.0,100.0,24.0
68372,2019-01-01 09:00:00,10.6,0.0,0.0,100.0,25.0


#### map schools to taxi zones

In [3]:
taxi_zones = gpd.read_file("/d/hpc/home/bv7063/bigdata/taxi_zones/taxi_zones.shp")
taxi_zones.head()

,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,geometry
0,1,0.116357,0.000782,Newark Airport,1,EWR,"POLYGON ((933100.918 192536.086, 933091.011 19..."
1,2,0.433470,0.004866,Jamaica Bay,2,Queens,"MULTIPOLYGON (((1033269.244 172126.008, 103343..."
2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((1026308.77 256767.698, 1026495.593 2..."
3,4,0.043567,0.000112,Alphabet City,4,Manhattan,"POLYGON ((992073.467 203714.076, 992068.667 20..."
4,5,0.092146,0.000498,Arden Heights,5,Staten Island,"POLYGON ((935843.31 144283.336, 936046.565 144..."


In [4]:
print(taxi_zones.crs)
print(taxi_zones.columns)


EPSG:2263
Index(['OBJECTID', 'Shape_Leng', 'Shape_Area', 'zone', 'LocationID', 'borough',
       'geometry'],
      dtype='str')


In [5]:
schools = pd.read_csv("/d/hpc/home/bv7063/bigdata/NYS_Schools_-7077623572626722609.csv")

schools_gdf = gpd.GeoDataFrame(
    schools,
    geometry=gpd.points_from_xy(schools["x"], schools["y"]),
    crs="EPSG:26918"
)

schools_gdf = schools_gdf.to_crs("EPSG:2263")


In [6]:
# spatially join taxi to zones
schools_with_zones = gpd.sjoin(
    schools_gdf,
    taxi_zones[["LocationID", "zone", "borough", "geometry"]],
    how="inner",
    predicate="within"
)


In [8]:
schools_with_zones[[
    "School Name",
    "City",
    "County",
    "Institute Type Description",
    "Institute Sub Type Description",
    "LocationID",
    "zone",
    "borough"
]].head(100)


,School Name,City,County,Institute Type Description,Institute Sub Type Description,LocationID,zone,borough
0,NEW YORK FILM ACADEMY,NEW YORK,NEW YORK,REGENTS APPROVED PROPRIETARY COLLEGES,4 YEAR PROPRIETARY,88,Financial District South,Manhattan
4,CITY UNIVERSITY OF NEW YORK SCHOOL OF LAW,FLUSHING,QUEENS,CUNY,CUNY GRADUATE CENTER,135,Kew Gardens Hills,Queens
7,SWEDISH INSTITUTE INC,NEW YORK,NEW YORK,REGENTS APPROVED PROPRIETARY COLLEGES,4 YEAR PROPRIETARY,186,Penn Station/Madison Sq West,Manhattan
9,TOURO UNIVERSITY,NEW YORK,NEW YORK,REGENTS APPROVED INDEPENDENT COLLEGES,4-YEAR INDEPENDENT,230,Times Sq/Theatre District,Manhattan
10,BANK STREET COLLEGE OF EDUCATION-LONG ISLAND CITY,LONG ISLAND CITY,QUEENS,REGENTS APPROVED INDEPENDENT COLLEGES,2 YEAR INDEPENDENT,145,Long Island City/Hunters Point,Queens
...,...,...,...,...,...,...,...,...
233,SUNY MARITIME COLLEGE,BRONX,BRONX,SUNY,SUNY SPECIALIZED COLLEGES,208,Schuylerville/Edgewater Park,Bronx
251,LONG ISLAND UNIVERSITY - NEW YORK UNIVERSITY C...,NEW YORK,NEW YORK,REGENTS APPROVED INDEPENDENT COLLEGES,4-YEAR INDEPENDENT,114,Greenwich Village South,Manhattan
253,CITY UNIVERSITY OF NEW YORK HERBERT H. LEHMAN ...,BRONX,BRONX,CUNY,CUNY 4 YEAR COLLEGE,241,Van Cortlandt Village,Bronx
254,CUNY QUEENS COLLEGE,FLUSHING,QUEENS,CUNY,CUNY 4 YEAR COLLEGE,135,Kew Gardens Hills,Queens


In [25]:
zone_school_counts = (
    schools_with_zones.groupby("LocationID")
    .size()
    .reset_index(name="school_university_count")
)



In [26]:
zone_school_counts.head()


,LocationID,school_university_count
0,7,1
1,20,1
2,21,1
3,23,2
4,25,1


In [37]:
zone_school_counts["LocationID"] = zone_school_counts["LocationID"].astype("float64")
zone_school_counts_dd = dd.from_pandas(zone_school_counts, npartitions=1)


In [38]:
BASE_DIR = Path("/d/hpc/home/bv7063/bigdata/output")
taxi_weather_df = dd.read_parquet(BASE_DIR / "augmented_weather_parquet")


In [39]:
# merge onto pickup values
taxi_augmented_df = taxi_weather_df.merge(
    zone_school_counts_dd.rename(columns={
        "LocationID": "PULocationID",
        "school_university_count": "pickup_school_university_count"
    }),
    on="PULocationID",
    how="left"
)



In [40]:
# Merge onto drop off locations
taxi_augmented_df = taxi_augmented_df.merge(
    zone_school_counts_dd.rename(columns={
        "LocationID": "DOLocationID",
        "school_university_count": "dropoff_school_university_count"
    }),
    on="DOLocationID",
    how="left"
)


In [41]:
taxi_augmented_df["pickup_school_university_count"] = (
    taxi_augmented_df["pickup_school_university_count"].fillna(0)
)

taxi_augmented_df["dropoff_school_university_count"] = (
    taxi_augmented_df["dropoff_school_university_count"].fillna(0)
)


In [42]:
taxi_augmented_df[[
    "PULocationID",
    "DOLocationID",
    "pickup_school_university_count",
    "dropoff_school_university_count"
]].head(20)



,PULocationID,DOLocationID,pickup_school_university_count,dropoff_school_university_count
0,151.0,239.0,0.0,1.0
1,239.0,246.0,1.0,0.0
2,163.0,229.0,0.0,0.0
3,229.0,7.0,0.0,1.0
4,141.0,234.0,1.0,1.0
5,246.0,162.0,0.0,1.0
6,238.0,151.0,0.0,0.0
7,163.0,25.0,0.0,1.0
8,224.0,25.0,0.0,1.0
9,141.0,234.0,1.0,1.0


In [43]:
taxi_augmented_df[[
    "PULocationID",
    "DOLocationID",
    "pickup_school_university_count",
    "dropoff_school_university_count",
]].head(20)


,PULocationID,DOLocationID,pickup_school_university_count,dropoff_school_university_count
0,151.0,239.0,0.0,1.0
1,239.0,246.0,1.0,0.0
2,163.0,229.0,0.0,0.0
3,229.0,7.0,0.0,1.0
4,141.0,234.0,1.0,1.0
5,246.0,162.0,0.0,1.0
6,238.0,151.0,0.0,0.0
7,163.0,25.0,0.0,1.0
8,224.0,25.0,0.0,1.0
9,141.0,234.0,1.0,1.0


In [44]:
# what we augment here 
# how many schools / universities are in the pickup / dropoff taxi zone
taxi_augmented_df[[
    "pickup_school_university_count",
    "dropoff_school_university_count",
]].isna().mean().compute()


pickup_school_university_count     0.0
dropoff_school_university_count    0.0
dtype: float64

In [45]:
taxi_augmented_df.to_parquet(
    BASE_DIR / "augmented_weather_schools_parquet",
    engine="pyarrow",
    write_index=False
)


In [47]:
# not needed create only lookup table 
zone_school_summary = (
    schools_with_zones.groupby(["LocationID", "zone", "borough"])["School Name"]
    .agg(lambda names: sorted(set(names)))
    .reset_index(name="school_university_names")
)

zone_school_summary["school_university_count"] = (
    zone_school_summary["school_university_names"].apply(len)
)

zone_school_summary = zone_school_summary[
    ["LocationID", "zone", "borough", "school_university_count", "school_university_names"]
]

zone_school_summary.head(20)



,LocationID,zone,borough,school_university_count,school_university_names
0,7,Astoria,Queens,1,[THE KINGS COLLEGE]
1,20,Belmont,Bronx,1,[FORDHAM UNIV (ROSE HILL-LINCOLN CTR)]
2,21,Bensonhurst East,Brooklyn,1,[ELYON COLLEGE]
3,23,Bloomfield/Emerson Hill,Staten Island,2,[CITY UNIVERSITY OF NEW YORK COLLEGE OF STATEN...
4,25,Boerum Hill,Brooklyn,1,[ST FRANCIS COLLEGE]
5,26,Borough Park,Brooklyn,1,[DAEMEN UNIVERSITY-BROOKLYN BRANCH]
6,33,Brooklyn Heights,Brooklyn,1,[BROOKLYN LAW SCHOOL]
7,40,Carroll Gardens,Brooklyn,1,[LONG ISLAND COLLEGE HOSPITAL SCH NURSING]
8,41,Central Harlem,Manhattan,1,[COLL NEW ROCHELLE ROSA PARKS CAMPUS]
9,49,Clinton Hill,Brooklyn,2,"[PRATT INSTITUTE, ST JOSEPH'S UNIVERSITY]"


In [48]:
zone_school_summary.to_csv(
    "/d/hpc/home/bv7063/bigdata/output/zone_school_summary.csv",
    index=False
)


#### Vicinity of major bussiness and attractions - note attracions missing 

In [49]:
BASE_DIR = Path("/d/hpc/home/bv7063/bigdata/output")
taxi_augmented_df = dd.read_parquet(BASE_DIR / "augmented_weather_schools_parquet")


In [ ]:
# load taxi zones
taxi_zones = gpd.read_file("/d/hpc/home/bv7063/bigdata/taxi_zones/taxi_zones.shp")
taxi_zones = taxi_zones[["LocationID", "zone", "borough", "geometry"]]

In [52]:
# load business csv 
businesses = pd.read_csv("/d/hpc/home/bv7063/bigdata/businesses_20260406.csv")
businesses.head()

,Business Name,Community Board,Council District,BIN,BBL,Census Tract (2010),Latitude,Longitude
0,"GREGORY ULTO, INC.",318.0,45.0,3217486.0,3.078250e+09,730.0,40.62446,-73.932304
1,YA.K.K INC,103.0,1.0,1001812.0,1.001640e+09,29.0,40.71573,-73.998788
2,"PROCESS SERVER PLUS, INC.",409.0,32.0,4188647.0,4.090710e+09,38.0,40.68463,-73.844767
3,BROADWAY BEER AND SMOKE INC.,401.0,22.0,4008258.0,4.006110e+09,59.0,40.76188,-73.925212
4,NEW YORK GINSENG CITY INC,407.0,20.0,4112315.0,4.049760e+09,871.0,40.75913,-73.831422


In [ ]:
# ensure all have lat and long
businesses = businesses.dropna(subset=["Longitude", "Latitude"])


In [54]:
businesses_gdf = gpd.GeoDataFrame(
    businesses,
    geometry=gpd.points_from_xy(businesses["Longitude"], businesses["Latitude"]),
    crs="EPSG:4326"
)


In [55]:
# reproject to taxi zones crs
businesses_gdf = businesses_gdf.to_crs(taxi_zones.crs)


In [56]:
businesses_with_zones = gpd.sjoin(
    businesses_gdf,
    taxi_zones,
    how="inner",
    predicate="within"
)


In [57]:
businesses_with_zones[[
    "Business Name",
    "LocationID",
    "zone",
    "borough"
]].head(20)


,Business Name,LocationID,zone,borough
0,"GREGORY ULTO, INC.",91,Flatlands,Brooklyn
1,YA.K.K INC,45,Chinatown,Manhattan
2,"PROCESS SERVER PLUS, INC.",258,Woodhaven,Queens
3,BROADWAY BEER AND SMOKE INC.,7,Astoria,Queens
4,NEW YORK GINSENG CITY INC,92,Flushing,Queens
7,"JAMES ROBINSON, INC.",163,Midtown North,Manhattan
8,JUBILEE VAPE & SMOKE INC,231,TriBeCa/Civic Center,Manhattan
9,guru kirpa farms inc,19,Bellerose,Queens
10,Amtel NY LLC,92,Flushing,Queens
12,QUEENS GOURMET DELI CORP,205,Saint Albans,Queens


In [59]:
zone_business_counts = (
    businesses_with_zones.groupby("LocationID")
    .size()
    .reset_index(name="business_count")
)

zone_business_counts.head(10)

,LocationID,business_count
0,3,192
1,4,48
2,5,67
3,6,101
4,7,506
5,9,105
6,10,153
7,11,204
8,12,1
9,13,50


In [65]:
# convert to dask 
zone_business_counts["LocationID"] = zone_business_counts["LocationID"].astype("float64")
zone_business_counts_dd = dd.from_pandas(zone_business_counts, npartitions=1)


In [68]:
# merge onto picup zone and dropoff zone

taxi_augmented_df = taxi_augmented_df.merge(
    zone_business_counts_dd.rename(columns={
        "LocationID": "PULocationID",
        "business_count": "pickup_business_count"
    }),
    on="PULocationID",
    how="left"
)

taxi_augmented_df = taxi_augmented_df.merge(
    zone_business_counts_dd.rename(columns={
        "LocationID": "DOLocationID",
        "business_count": "dropoff_business_count"
    }),
    on="DOLocationID",
    how="left"
)

In [69]:
taxi_augmented_df["pickup_business_count"] = taxi_augmented_df["pickup_business_count"].fillna(0)
taxi_augmented_df["dropoff_business_count"] = taxi_augmented_df["dropoff_business_count"].fillna(0)


In [70]:
taxi_augmented_df[[
    "PULocationID",
    "DOLocationID",
    "pickup_business_count",
    "dropoff_business_count"
]].head(20)


,PULocationID,DOLocationID,pickup_business_count,dropoff_business_count
0,151.0,239.0,91.0,163.0
1,239.0,246.0,163.0,112.0
2,163.0,229.0,182.0,193.0
3,229.0,7.0,193.0,506.0
4,141.0,234.0,186.0,220.0
5,246.0,162.0,112.0,190.0
6,238.0,151.0,164.0,91.0
7,163.0,25.0,182.0,155.0
8,224.0,25.0,15.0,155.0
9,141.0,234.0,186.0,220.0


In [71]:
taxi_augmented_df.to_parquet(
    BASE_DIR / "augmented_weather_schools_businesses_parquet",
    engine="pyarrow",
    write_index=False
)
